In [ ]:
import pandas as pd
import torch
from rdkit import Chem
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

print(torch.__version__)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from torch_geometric.utils import to_networkx
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

In [ ]:
data = pd.read_csv("../Dataset/qm8.csv")

In [ ]:
data = data.iloc[:100,:]

In [ ]:
def smiles_to_graph(smiles, y_labels=None):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    # Add hydrogens explicitly as they are often important in QM datasets
    mol = Chem.AddHs(mol)
    
    # Atom features: Atomic number
    atom_features = []
    for atom in mol.GetAtoms():
        atom_features.append([atom.GetAtomicNum()])
    x = torch.tensor(atom_features, dtype=torch.float)
    
    # Edge index: Bonds
    edge_index = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_index.append([i, j])
        edge_index.append([j, i])
    
    if not edge_index:
        edge_index = torch.empty((2, 0), dtype=torch.long)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    y = None
    if y_labels is not None:
        y = torch.tensor([y_labels], dtype=torch.float)
    
    return Data(x=x, edge_index=edge_index)

data['graph'] = data['smiles'].apply(smiles_to_graph)

In [ ]:
data.graph.head()

In [ ]:
data.graph[1]

In [ ]:
G = to_networkx(data['graph'][95], to_undirected=True)

pos = nx.spring_layout(G, dim=3, seed=0)

node_xyz = np.array([pos[v] for v in sorted(G)])
edge_xyz = np.array([(pos[v], pos[u]) for u, v in G.edges()])

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

for dim in (ax.xaxis, ax.yaxis, ax.zaxis):
    dim.set_ticks([])

ax.scatter(*node_xyz.T, s=500, c='#0A047A')

for vizedge in edge_xyz:
    ax.plot(*vizedge.T, color='k')

plt.show()

In [ ]:
from typing import Any
from torch.nn import Linear, ReLU, Sequential, BatchNorm1d, Dropout
import torch.nn.functional as F
from torch_geometric.nn import GINConv
from torch_geometric.nn import global_add_pool, global_mean_pool
import torch_geometric

class GIN(torch.nn.Module):
    """
    This class is an implementation of Graph Isomorphism Network (GIN) for graph-level regression tasks. The GIN architecture is designed to capture the structural information of graphs effectively, making it suitable for tasks like molecular property prediction.
    """
    def __init__(self, dim_h, dataset: torch_geometric.data.Data):
        super(GIN, self).__init__()

        def gin_mlp(in_d, out_d):
            return Sequential(
                Linear(in_d, out_d),
                BatchNorm1d(out_d),
                ReLU(),
                Linear(out_d, out_d),
                ReLU()
            )

        self.conv1 = GINConv(gin_mlp(dataset.num_node_features, dim_h))
        self.conv2 = GINConv(gin_mlp(dim_h, dim_h))
        self.conv3 = GINConv(gin_mlp(dim_h, dim_h))

        self.post_readout = Sequential(
            Linear(dim_h, dim_h),
            ReLU(),
            Dropout(p=0.5),
            Linear(dim_h, 12),
        )

    def forward(self, data: torch_geometric.data.Data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # Node embeddings
        h1 = self.conv1(x, edge_index)
        h2 = self.conv2(h1, edge_index)
        h3 = self.conv3(h2, edge_index)

        graph_embed = global_add_pool(h3, batch)

        return self.post_readout(graph_embed)

In [ ]:
class TrainingTesting(GIN):
    """
    A comprehensive class for training and testing the GIN model on a given dataset.
    This class encapsulates the entire workflow, including data loading, model initialization, training loop, and evaluation metrics.
    It is designed to be flexible and can be easily adapted to different datasets and configurations.
    """
    def __init__(self, dataset: torch_geometric.data.Data):
        super().__init__(dataset.num_node_features, dataset.num_node_labels)
        self.dataset = dataset
        self.epochs = 50
        self.optimizer = torch.optim.Adam(self.parameters(),
                                          lr=0.001,
                                          weight_decay=0.0005)
        self.criterion = torch.nn.MSELoss()
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(self.optimizer,
                                                                    mode='min',
                                                                    factor=0.5,
                                                                    patience=10,
                                                                    verbose=True)
    def training(self, dataset: Any) -> None:
        """
        This method handles the training loop for the GIN model. It iterates over the training dataset, computes the loss, and updates the model parameters using backpropagation.
        The method also includes functionality for tracking training progress and can be extended to include features like early stopping or learning rate scheduling.

        :param dataset:
        :return:
        """
        self.train()

